# XAS from single run with shutter-cycled background (Kaiyu)

This notebook is for the new acquisition pattern inside one nominal-energy scan run:

1. One run (for example 58780) contains many nominal photon-energy points.
2. For each nominal-energy point: about 30 s background (shutter closed), then about 2 s x-ray signal (shutter open).
3. Compute XAS point-by-point with `XAS = sum(GMD) / sum(VLS)` after subtracting per-bunch VLS background from the paired background window.

The notebook also inspects raw H5 metadata and tries to locate:
- nominal photon energy
- hall GMD
- tunnel GMD

without hard-coding one exact machine path.

In [1]:
import sys
import glob
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt

# Resolve repo root robustly from notebook location or repo root cwd.
cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "analysis" / "scripts").exists():
        repo_root = p
        break
if repo_root is None:
    raise RuntimeError("Could not locate repository root containing analysis/scripts")

sys.path.insert(0, str(repo_root / "analysis" / "scripts"))

import config
from data_loading import load_raw_h5

%matplotlib inline

## 1) Configuration (edit first)

In [2]:
# Target run
RUN_NO = 58780
MAX_FILES = None          # None = all files for this run

# VLS preprocessing
ROI = (500, 600)          # Set None for full pixel axis
SIGNAL_BUNCH_RANGE = (0, 60)

# Grouping and quality controls
NOMINAL_ENERGY_TOL_EV = 0.05   # split energy plateaus when |dE| > tol
GMD_LO = 0.0                   # set >0 if you want to reject tiny/negative pulses
GMD_HI = None

# Data-driven signal/background detection (per nominal-energy segment).
# Signal is found from an adaptive pixel ROI around the strongest train-to-train contrast.
ADAPTIVE_HALF_WIDTH_PX = 20
PEAK_Q_LOW = 20.0
PEAK_Q_HIGH = 95.0

# 1D two-cluster detector on per-train scores; these bounds protect against degenerate splits.
MIN_SIGNAL_FRACTION = 0.01
MAX_SIGNAL_FRACTION = 0.50

# If clustering is degenerate, fallback to top fraction by score.
FALLBACK_SIGNAL_FRACTION = 0.08

# Background for each detected signal train: local median of nearby detected background trains.
BG_NEIGHBOR_TRAINS = 80

# Optional timing info (for reference only, not used for hard slicing).
EST_BG_SECONDS = 30.0
EST_SIG_SECONDS = 2.0
EST_TRAIN_RATE_HZ = 10.0

# Fallback dataset path candidates (used only if auto-detection cannot decide).
NOMINAL_ENERGY_CANDIDATES = [
    "/FL2/Photon Diagnostic/Wavelength/OPIS tunnel/Processed/nominal photon energy/value",
    "/FL2/Photon Diagnostic/Wavelength/OPIS tunnel/Processed/setpoint photon energy/value",
    "/FL2/Photon Diagnostic/Wavelength/OPIS tunnel/Processed/mean photon energy/value",
]
NOMINAL_ENERGY_INDEX_CANDIDATES = [
    "/FL2/Photon Diagnostic/Wavelength/OPIS tunnel/Processed/nominal photon energy/index",
    "/FL2/Photon Diagnostic/Wavelength/OPIS tunnel/Processed/setpoint photon energy/index",
    "/FL2/Photon Diagnostic/Wavelength/OPIS tunnel/Processed/mean photon energy/index",
]
HALL_GMD_VALUE_PATH = "/FL2/Photon Diagnostic/GMD/Pulse resolved energy/energy hall/value"
HALL_GMD_INDEX_PATH = "/FL2/Photon Diagnostic/GMD/Pulse resolved energy/energy hall/index"
TUNNEL_GMD_CANDIDATES = [
    "/FL2/Photon Diagnostic/GMD/Pulse resolved energy/energy tunnel/value",
    "/FL2/Photon Diagnostic/GMD/Pulse resolved energy/tunnel/value",
]
TUNNEL_GMD_INDEX_CANDIDATES = [
    "/FL2/Photon Diagnostic/GMD/Pulse resolved energy/energy tunnel/index",
    "/FL2/Photon Diagnostic/GMD/Pulse resolved energy/tunnel/index",
]

print(f"RUN_NO={RUN_NO}, MAX_FILES={MAX_FILES}")
print(f"ROI={ROI}, SIGNAL_BUNCH_RANGE={SIGNAL_BUNCH_RANGE}")
print(
    f"Detection: ROI half-width={ADAPTIVE_HALF_WIDTH_PX}px, "
    f"signal fraction bounds=[{MIN_SIGNAL_FRACTION:.3f}, {MAX_SIGNAL_FRACTION:.3f}]"
)
print(
    f"Timing hint only: bg~{EST_BG_SECONDS}s, signal~{EST_SIG_SECONDS}s at ~{EST_TRAIN_RATE_HZ} Hz"
)

RUN_NO=58780, MAX_FILES=None
ROI=(500, 600), SIGNAL_BUNCH_RANGE=(0, 60)
Detection: ROI half-width=20px, signal fraction bounds=[0.010, 0.500]
Timing hint only: bg~30.0s, signal~2.0s at ~10.0 Hz


## 2) Locate run files and inspect available raw parameters

In [3]:
raw_dir = Path(config.RAW_H5_DIR)
pattern = str(raw_dir / f"*run{RUN_NO}*.h5")
h5_paths = sorted(glob.glob(pattern))
if MAX_FILES is not None:
    h5_paths = h5_paths[:MAX_FILES]

print(f"raw_dir: {raw_dir}")
print(f"pattern: {pattern}")
print(f"files found: {len(h5_paths)}")
for p in h5_paths[:5]:
    print("  " + Path(p).name)
if not h5_paths:
    print("No files found yet. Download the run and re-run this cell.")

raw_dir: /Users/hukaiyu/Desktop/PhD/FLASH Beamtime 202605/glycine26/11022188/raw/hdf/online-0/fl2user1
pattern: /Users/hukaiyu/Desktop/PhD/FLASH Beamtime 202605/glycine26/11022188/raw/hdf/online-0/fl2user1/*run58780*.h5
files found: 0
No files found yet. Download the run and re-run this cell.


In [ ]:
def _path_exists(h5, path):
    return path in h5

def _pick_existing(h5, candidates):
    for p in candidates:
        if p in h5:
            return p
    return None

def _print_dataset_info(h5, path, label):
    if path is None:
        print(f"{label:20s}: NOT FOUND")
        return
    ds = h5[path]
    print(f"{label:20s}: {path}")
    print(f"{'':20s}  shape={ds.shape}, dtype={ds.dtype}")

if h5_paths:
    with h5py.File(h5_paths[0], "r") as f:
        nominal_val_path = _pick_existing(f, NOMINAL_ENERGY_CANDIDATES)
        nominal_idx_path = _pick_existing(f, NOMINAL_ENERGY_INDEX_CANDIDATES)
        tunnel_val_path = _pick_existing(f, TUNNEL_GMD_CANDIDATES)
        tunnel_idx_path = _pick_existing(f, TUNNEL_GMD_INDEX_CANDIDATES)

        print("Detected key raw datasets in first file:")
        _print_dataset_info(f, HALL_GMD_VALUE_PATH if _path_exists(f, HALL_GMD_VALUE_PATH) else None, "hall_gmd value")
        _print_dataset_info(f, HALL_GMD_INDEX_PATH if _path_exists(f, HALL_GMD_INDEX_PATH) else None, "hall_gmd index")
        _print_dataset_info(f, tunnel_val_path, "tunnel_gmd value")
        _print_dataset_info(f, tunnel_idx_path, "tunnel_gmd index")
        _print_dataset_info(f, nominal_val_path, "nominal_energy value")
        _print_dataset_info(f, nominal_idx_path, "nominal_energy index")

        # Keep choices in variables for later cells.
        NOMINAL_VALUE_PATH = nominal_val_path
        NOMINAL_INDEX_PATH = nominal_idx_path
        TUNNEL_GMD_VALUE_PATH = tunnel_val_path
        TUNNEL_GMD_INDEX_PATH = tunnel_idx_path

        print()
        print("Quick browse under /FL2/Photon Diagnostic:")
        if "/FL2/Photon Diagnostic" in f:
            grp = f["/FL2/Photon Diagnostic"]
            for k in grp.keys():
                print("  " + k)
        else:
            print("  group not found in this file")
else:
    NOMINAL_VALUE_PATH = None
    NOMINAL_INDEX_PATH = None
    TUNNEL_GMD_VALUE_PATH = None
    TUNNEL_GMD_INDEX_PATH = None

## 3) Helper functions for train-ID alignment of raw streams

In [ ]:
def align_by_tid(src_tid, master_tid):
    pos = np.searchsorted(src_tid, master_tid)
    in_range = pos < len(src_tid)
    matched = np.zeros(master_tid.shape, dtype=bool)
    matched[in_range] = src_tid[pos[in_range]] == master_tid[in_range]
    return matched, pos[matched]

def extract_aligned_raw_stream(h5_paths, index_path, value_path, master_tid, *, keep_channel=None):
    """
    Return array aligned to master_tid by train ID.

    Supported value shapes:
    - (n_trains,)                -> output (n_master,)
    - (n_trains, n_channels, m)  -> choose channel keep_channel, output (n_master, m)
    - (n_trains, m)              -> output (n_master, m)
    """
    if (index_path is None) or (value_path is None):
        return None

    out = None
    for fp in h5_paths:
        with h5py.File(fp, "r") as f:
            if (index_path not in f) or (value_path not in f):
                continue
            idx = f[index_path][...]
            val = f[value_path][...]

            if val.ndim == 3:
                ch = 0 if keep_channel is None else int(keep_channel)
                val2 = val[:, ch, :]
            elif val.ndim == 2:
                val2 = val
            elif val.ndim == 1:
                val2 = val
            else:
                raise ValueError(f"Unsupported ndim={val.ndim} for {value_path}")

            if out is None:
                shape_tail = val2.shape[1:]
                out_shape = (master_tid.shape[0],) + shape_tail
                out = np.full(out_shape, np.nan, dtype=np.float32)

            matched, src_pos = align_by_tid(idx, master_tid)
            if matched.any():
                out[matched] = val2[src_pos].astype(np.float32)

    return out

## 4) Load run and collect hall/tunnel GMD + nominal photon energy

In [ ]:
if not h5_paths:
    raise RuntimeError("No run files found. Download raw data first, then re-run.")

data = load_raw_h5(RUN_NO, config=2, max_files=MAX_FILES)
if ROI is not None:
    data = data.crop_vls(*ROI)

master_tid = data.tID.astype(np.float64)
hall_gmd = data.gmd.copy()   # from existing loader (energy hall, channel 0)

tunnel_gmd = extract_aligned_raw_stream(
    h5_paths,
    TUNNEL_GMD_INDEX_PATH,
    TUNNEL_GMD_VALUE_PATH,
    master_tid,
    keep_channel=0,
)

nominal_raw = extract_aligned_raw_stream(
    h5_paths,
    NOMINAL_INDEX_PATH,
    NOMINAL_VALUE_PATH,
    master_tid,
)

if nominal_raw is None:
    # Fallback: use OPIS mean photon energy from load_raw_h5
    nominal_train = data.mpe.astype(np.float32)
    nominal_source = "fallback: data.mpe (mean photon energy)"
else:
    if nominal_raw.ndim == 1:
        nominal_train = nominal_raw.astype(np.float32)
    else:
        # If a bunch-resolved array exists, average to one value per train.
        nominal_train = np.nanmean(nominal_raw, axis=1).astype(np.float32)
    nominal_source = NOMINAL_VALUE_PATH

print(f"VLS shape: {data.vls.shape}")
print(f"Hall GMD shape: {hall_gmd.shape}")
print(f"Tunnel GMD shape: {None if tunnel_gmd is None else tunnel_gmd.shape}")
print(f"Nominal energy source: {nominal_source}")
print(f"Nominal train values: finite={np.isfinite(nominal_train).sum()} / {nominal_train.size}")

## 5) Detect signal/background automatically per nominal-energy segment

For each nominal-energy segment, detection is data-driven (no fixed 30 s / 2 s slicing):

1. Build per-train spectra (mean over bunches in `SIGNAL_BUNCH_RANGE`).
2. Find a segment-specific peak pixel from train-to-train contrast (`q95 - q20`), so the ROI follows horizontal shifts with photon energy.
3. Build per-train score by integrating that adaptive ROI.
4. Split trains into two score clusters (background vs signal).
5. For each detected signal train, subtract the local median of nearby detected background trains.

This handles shifted signal position and imperfect timing.

In [ ]:
def contiguous_energy_segments(e_train, tol_ev):
    finite = np.isfinite(e_train)
    idx = np.where(finite)[0]
    if idx.size == 0:
        return []
    segs = []
    s = idx[0]
    prev = idx[0]
    for i in idx[1:]:
        if (i != prev + 1) or (abs(float(e_train[i]) - float(e_train[prev])) > tol_ev):
            segs.append((s, prev + 1))
            s = i
        prev = i
    segs.append((s, prev + 1))
    return segs


def two_means_1d(values, max_iter=40):
    """Simple 1D k=2 clustering without external dependencies."""
    v = np.asarray(values, dtype=float)
    finite = np.isfinite(v)
    labels = np.full(v.shape, -1, dtype=int)
    if finite.sum() < 2:
        return labels, np.nan, np.nan

    x = v[finite]
    c0, c1 = np.percentile(x, [25, 75])
    if not np.isfinite(c0) or not np.isfinite(c1) or c0 == c1:
        c0, c1 = float(np.nanmin(x)), float(np.nanmax(x))
        if c0 == c1:
            return labels, c0, c1

    for _ in range(max_iter):
        d0 = np.abs(x - c0)
        d1 = np.abs(x - c1)
        lab = (d1 < d0).astype(int)
        if np.all(lab == 0) or np.all(lab == 1):
            break
        nc0 = float(np.mean(x[lab == 0]))
        nc1 = float(np.mean(x[lab == 1]))
        if np.isclose(nc0, c0) and np.isclose(nc1, c1):
            c0, c1 = nc0, nc1
            break
        c0, c1 = nc0, nc1

    labels[finite] = lab
    return labels, c0, c1


segs = contiguous_energy_segments(nominal_train, NOMINAL_ENERGY_TOL_EV)
print(f"Found {len(segs)} nominal-energy contiguous segments")

records = []
detection_debug = []
b0, b1 = SIGNAL_BUNCH_RANGE
if not (0 <= b0 < b1 <= data.n_bunches):
    raise ValueError(f"SIGNAL_BUNCH_RANGE {SIGNAL_BUNCH_RANGE} invalid for n_bunches={data.n_bunches}")

px = data.vls_pixels.astype(np.float32)
all_nom = []
all_vls = []
all_hall = []
all_tunnel = []
all_seg = []

for seg_i, (s, e) in enumerate(segs):
    n_seg = e - s
    if n_seg < 5:
        continue

    seg_vls = data.vls[s:e, b0:b1, :]                   # (n_seg, n_bunch, n_px)
    seg_spec = np.nanmean(seg_vls, axis=1)              # (n_seg, n_px)

    # Adaptive ROI from train-to-train contrast, robust to baseline drift.
    q_lo = np.nanpercentile(seg_spec, PEAK_Q_LOW, axis=0)
    q_hi = np.nanpercentile(seg_spec, PEAK_Q_HIGH, axis=0)
    contrast = q_hi - q_lo
    if not np.isfinite(contrast).any():
        continue
    peak_px = int(np.nanargmax(contrast))
    lo = max(0, peak_px - ADAPTIVE_HALF_WIDTH_PX)
    hi = min(seg_spec.shape[1], peak_px + ADAPTIVE_HALF_WIDTH_PX + 1)

    # Per-train score used for background/signal classification.
    train_score = np.nansum(seg_spec[:, lo:hi], axis=1)

    labels, c0, c1 = two_means_1d(train_score)
    if np.isnan(c0) or np.isnan(c1):
        continue
    sig_label = int(np.nanargmax([c0, c1]))
    sig_mask = labels == sig_label
    frac_sig = float(np.mean(sig_mask))

    # Degenerate split guard: fallback to top-score fraction.
    if (frac_sig < MIN_SIGNAL_FRACTION) or (frac_sig > MAX_SIGNAL_FRACTION):
        n_top = max(1, int(round(FALLBACK_SIGNAL_FRACTION * n_seg)))
        order = np.argsort(train_score)
        sig_mask = np.zeros(n_seg, dtype=bool)
        sig_mask[order[-n_top:]] = True
        frac_sig = float(np.mean(sig_mask))

    bg_mask = ~sig_mask
    bg_idx = np.where(bg_mask)[0]
    sig_idx = np.where(sig_mask)[0]
    if bg_idx.size == 0 or sig_idx.size == 0:
        continue

    # Persist debug traces for plotting / tuning.
    detection_debug.append({
        "segment": seg_i,
        "train_start": s,
        "train_end": e,
        "peak_px": int(peak_px),
        "roi_lo": int(lo),
        "roi_hi": int(hi),
        "score": train_score.copy(),
        "sig_mask": sig_mask.copy(),
    })

    # Build shot-level arrays by subtracting local background per detected signal train.
    seg_shots = 0
    for t_local in sig_idx:
        left = max(0, t_local - BG_NEIGHBOR_TRAINS)
        right = min(n_seg, t_local + BG_NEIGHBOR_TRAINS + 1)
        local_bg = bg_idx[(bg_idx >= left) & (bg_idx < right)]
        if local_bg.size == 0:
            local_bg = bg_idx

        bg2d = np.nanmedian(data.vls[s + local_bg, :, :], axis=0)          # (n_bunch, n_px)
        sig2d = data.vls[s + t_local, b0:b1, :] - bg2d[b0:b1, :]            # (n_bunch_sig, n_px)
        vls_sum = np.nansum(sig2d, axis=1)
        g_hall = hall_gmd[s + t_local, b0:b1]
        if tunnel_gmd is None:
            g_tunnel = np.full_like(g_hall, np.nan, dtype=np.float32)
        else:
            g_tunnel = tunnel_gmd[s + t_local, b0:b1]

        good_hall = np.isfinite(g_hall) & np.isfinite(vls_sum)
        if GMD_LO is not None:
            good_hall &= g_hall >= GMD_LO
        if GMD_HI is not None:
            good_hall &= g_hall <= GMD_HI

        good_tunnel = np.isfinite(g_tunnel) & np.isfinite(vls_sum)
        if GMD_LO is not None:
            good_tunnel &= g_tunnel >= GMD_LO
        if GMD_HI is not None:
            good_tunnel &= g_tunnel <= GMD_HI

        if np.any(good_hall):
            all_nom.append(np.full(np.sum(good_hall), float(np.nanmean(nominal_train[s:e])), dtype=np.float32))
            all_vls.append(vls_sum[good_hall].astype(np.float32))
            all_hall.append(g_hall[good_hall].astype(np.float32))
            all_tunnel.append(g_tunnel[good_hall].astype(np.float32))
            all_seg.append(np.full(np.sum(good_hall), seg_i, dtype=np.int32))

        seg_shots += int(np.sum(good_hall))

    if len(all_vls) == 0:
        continue

    # Segment-level XAS summary from accumulated shots in this segment.
    seg_nom = float(np.nanmean(nominal_train[s:e]))
    seg_idx = np.where(np.concatenate(all_seg) == seg_i)[0] if all_seg else np.array([], dtype=int)
    if seg_idx.size == 0:
        continue

    v_seg = np.concatenate(all_vls)[seg_idx]
    h_seg = np.concatenate(all_hall)[seg_idx]
    t_seg = np.concatenate(all_tunnel)[seg_idx]

    xas_hall = np.nan
    if np.abs(np.nansum(v_seg)) > 1e-12:
        xas_hall = float(np.nansum(h_seg) / np.nansum(v_seg))
    xas_tunnel = np.nan
    good_t = np.isfinite(t_seg)
    if good_t.any() and np.abs(np.nansum(v_seg[good_t])) > 1e-12:
        xas_tunnel = float(np.nansum(t_seg[good_t]) / np.nansum(v_seg[good_t]))

    records.append({
        "segment": seg_i,
        "train_start": s,
        "train_end": e,
        "n_trains": n_seg,
        "nominal_e_mean": seg_nom,
        "nominal_e_std": float(np.nanstd(nominal_train[s:e])),
        "n_shots_hall": int(seg_idx.size),
        "n_shots_tunnel": int(np.sum(np.isfinite(t_seg))),
        "xas_hall": xas_hall,
        "xas_tunnel": xas_tunnel,
        "peak_px": int(peak_px),
        "roi_lo": int(lo),
        "roi_hi": int(hi),
        "signal_fraction": frac_sig,
    })

if not all_vls:
    print("No usable detected signal shots. Tune detection parameters in config.")
else:
    shot_nom = np.concatenate(all_nom)
    shot_vls = np.concatenate(all_vls)
    shot_hall = np.concatenate(all_hall)
    shot_tunnel = np.concatenate(all_tunnel)
    shot_seg = np.concatenate(all_seg)

print(f"Usable segments: {len(records)}")
if records:
    frac = np.array([r["signal_fraction"] for r in records], dtype=float)
    print(f"Detected signal-train fraction per segment: {np.nanmin(frac):.3f} .. {np.nanmax(frac):.3f}")

## 6) Results table and XAS plots

In [ ]:
if not records:
    raise RuntimeError("No results to plot.")

# Sort by nominal photon energy for plotting the scan.
records_sorted = sorted(records, key=lambda r: r["nominal_e_mean"])

print(f"{'seg':>4s} {'E_nom(eV)':>12s} {'E_std':>9s} {'n_hall':>8s} {'XAS_hall':>12s} {'n_tunnel':>9s} {'XAS_tunnel':>12s}")
print('-' * 80)
for r in records_sorted:
    print(
        f"{r['segment']:4d} {r['nominal_e_mean']:12.4f} {r['nominal_e_std']:9.4f} "
        f"{r['n_shots_hall']:8d} {r['xas_hall']:12.6g} {r['n_shots_tunnel']:9d} {r['xas_tunnel']:12.6g}"
    )

E = np.array([r['nominal_e_mean'] for r in records_sorted], dtype=float)
X_hall = np.array([r['xas_hall'] for r in records_sorted], dtype=float)
X_tunnel = np.array([r['xas_tunnel'] for r in records_sorted], dtype=float)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(E, X_hall, 'o-', lw=1.5, label='XAS using hall GMD')
if np.isfinite(X_tunnel).any():
    ax.plot(E, X_tunnel, 's--', lw=1.2, label='XAS using tunnel GMD')
ax.set_xlabel('Nominal photon energy (eV)')
ax.set_ylabel('XAS = sum(GMD) / sum(VLS)')
ax.set_title(f'Run {RUN_NO}: shutter-cycled XAS from one run')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 6b) XAS vs nominal photon energy (standalone plot)

Run this cell if you only want the XAS-vs-nominal-energy figure.

In [ ]:
if not records:
    raise RuntimeError("No XAS records available. Run Cell 13 first.")

records_sorted = sorted(records, key=lambda r: r["nominal_e_mean"])
E = np.array([r["nominal_e_mean"] for r in records_sorted], dtype=float)
X_hall = np.array([r["xas_hall"] for r in records_sorted], dtype=float)
X_tunnel = np.array([r["xas_tunnel"] for r in records_sorted], dtype=float)

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.plot(E, X_hall, "o-", lw=1.6, color="mediumseagreen", label="XAS (hall GMD)")
if np.isfinite(X_tunnel).any():
    ax.plot(E, X_tunnel, "s--", lw=1.3, color="steelblue", label="XAS (tunnel GMD)")

for r in records_sorted:
    ax.annotate(
        f"seg {r['segment']}",
        (r["nominal_e_mean"], r["xas_hall"]),
        textcoords="offset points",
        xytext=(4, 4),
        fontsize=8,
    )

ax.set_xlabel("Nominal photon energy (eV)")
ax.set_ylabel("XAS = sum(GMD) / sum(VLS)")
ax.set_title(f"Run {RUN_NO}: XAS vs nominal photon energy")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

## 7) Optional diagnostics for segmentation

In [ ]:
if not records:
    raise RuntimeError("No detection diagnostics available.")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(nominal_train, lw=0.8)
axes[0].set_ylabel('Nominal E (eV)')
axes[0].set_title('Per-train nominal photon energy')

hall_train_mean = np.nanmean(hall_gmd, axis=1)
axes[1].plot(hall_train_mean, lw=0.8, color='tab:green')
axes[1].set_ylabel('Hall GMD mean per train (uJ)')
axes[1].set_xlabel('Train index in loaded run')

for (s, e) in [(r['train_start'], r['train_end']) for r in records]:
    axes[0].axvspan(s, e, color='gray', alpha=0.1)
    axes[1].axvspan(s, e, color='gray', alpha=0.1)

fig.tight_layout()
plt.show()

# Show detector behavior for one segment (score split and adaptive ROI).
dbg = detection_debug[0]
tx = np.arange(dbg['train_start'], dbg['train_end'])
score = dbg['score']
sig = dbg['sig_mask']

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(tx, score, color='k', lw=1.0, label='train score (adaptive ROI integral)')
ax.scatter(tx[sig], score[sig], s=18, color='tab:red', label='detected signal trains')
ax.scatter(tx[~sig], score[~sig], s=12, color='tab:blue', alpha=0.7, label='detected background trains')
ax.set_xlabel('Train index')
ax.set_ylabel('Score (arb.)')
ax.set_title(
    f"Detection debug: segment {dbg['segment']}  "
    f"ROI=[{dbg['roi_lo']},{dbg['roi_hi']}) around peak pixel {dbg['peak_px']}"
)
ax.grid(alpha=0.3)
ax.legend(loc='best')
fig.tight_layout()
plt.show()